# 02 — Data Cleaning

This notebook prepares the raw SPY market data for exploratory analysis,
feature engineering, and volatility prediction.

The raw data were downloaded in `01_data_loading.ipynb` and stored in
`data/raw/spy_daily.csv`.

The cleaning process will focus on:

- Loading the raw dataset.
- Inspecting its structure and index.
- Ensuring the date index is correctly formatted and ordered.
- Checking for duplicate observations.
- Checking for missing values and invalid observations.
- Applying only the necessary cleaning operations.

No predictive features or target variables are constructed in this notebook.

## 1. Load The Raw Data

### 1.1: Import Libraries

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

### 1.2: Define Paths

In [2]:
# Define project directories

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DATA_FILE = RAW_DATA_DIR / "spy_daily.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data file: {RAW_DATA_FILE}")

Project root: /Users/amitkumar/Desktop/Computing/Projects/Finance/market-volatility
Raw data file: /Users/amitkumar/Desktop/Computing/Projects/Finance/market-volatility/data/raw/spy_daily.csv


### 1.3: Load the raw SPY data

In [4]:
# Load the raw SPY data

spy_data = pd.read_csv(
    RAW_DATA_FILE,
    index_col=0, #Tells pandas to use that first column as the index
    parse_dates=True #converts it back into a proper datetime index
)

spy_data.head()

,Close,High,Low,Open,Volume
2010-01-04,84.578491,84.623268,83.220220,83.862041,118944600
2010-01-05,84.802376,84.839693,84.220261,84.526247,111579900
2010-01-06,84.862091,85.071055,84.653127,84.720292,116074400
2010-01-07,85.220306,85.324788,84.466539,84.705356,131091100
2010-01-08,85.503906,85.541224,84.824774,84.996420,126402800


### 1.4: Inspect the Index
The date index is central to the time-series analysis in this project.
We therefore verify that it is a proper `DatetimeIndex`, that the
observations are ordered chronologically, and that the index contains
no duplicate dates.

In [5]:
# Inspect the date index

print(f"Index type: {type(spy_data.index)}")
print(f"Index is sorted: {spy_data.index.is_monotonic_increasing}")
print(f"Duplicate dates: {spy_data.index.duplicated().sum()}")

Index type: <class 'pandas.DatetimeIndex'>
Index is sorted: True
Duplicate dates: 0


### 1.5: Check Data Types and Missing Values
We now check the data types of the market variables and verify that no
missing observations were introduced when the raw data were saved and
reloaded.

In [6]:
# Inspect data types
spy_data.dtypes

Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

In [7]:
# Check for missing values

spy_data.isna().sum()

Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [9]:
# Check for infinite values

np.isinf(spy_data.select_dtypes(include=np.number)).sum()

Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

### 1.6: Validate OHLC Relationships
For each trading day, the reported high price should be greater than or
equal to both the open and close prices, while the low price should be less
than or equal to both.

We check these relationships before proceeding with the dataset.

In [10]:
# Check OHLC price relationships

invalid_high = (
    (spy_data["High"] < spy_data["Open"]) |
    (spy_data["High"] < spy_data["Close"])
)

invalid_low = (
    (spy_data["Low"] > spy_data["Open"]) |
    (spy_data["Low"] > spy_data["Close"])
)

print(f"Rows where High is below Open or Close: {invalid_high.sum()}")
print(f"Rows where Low is above Open or Close: {invalid_low.sum()}")

Rows where High is below Open or Close: 1
Rows where Low is above Open or Close: 0


In [11]:
# Inspect the row(s) violating the High >= Open, Close condition

spy_data.loc[invalid_high, ["Open", "High", "Low", "Close"]]

,Open,High,Low,Close
2018-01-11,241.204106,242.40683,241.037302,242.40683


In [12]:
# Calculate the size of the violation

spy_data.loc[invalid_high, "High"] - spy_data.loc[invalid_high, ["Open", "Close"]].max(axis=1)

2018-01-11   -5.684342e-14
dtype: float64

#### Observation

The strict validation identified one apparent violation:
$
-5.684342\times10^{-14},
$
which is effectively zero at the scale of the prices.

For 2018-01-11, the High and Close values are equal to the displayed
precision. The tiny discrepancy arises from floating-point representation.

Therefore, there is no genuine data-quality problem. We retain the original
data and use a small numerical tolerance for the validation check.

In [13]:
# Check OHLC price relationships with a numerical tolerance

TOLERANCE = 1e-10

invalid_high = (
    (spy_data["High"] < spy_data["Open"] - TOLERANCE) |
    (spy_data["High"] < spy_data["Close"] - TOLERANCE)
)

invalid_low = (
    (spy_data["Low"] > spy_data["Open"] + TOLERANCE) |
    (spy_data["Low"] > spy_data["Close"] + TOLERANCE)
)

print(f"Rows where High is below Open or Close: {invalid_high.sum()}")
print(f"Rows where Low is above Open or Close: {invalid_low.sum()}")

Rows where High is below Open or Close: 0
Rows where Low is above Open or Close: 0


### 1.7: Check for Non-Positive Values
Prices and trading volume should be strictly positive. We therefore check
for any observations containing zero or negative values.

In [14]:
# Check for non-positive values

price_columns = ["Open", "High", "Low", "Close"]

for column in price_columns:
    print(f"{column}: {(spy_data[column] <= 0).sum()} non-positive values")

print(f"Volume: {(spy_data['Volume'] <= 0).sum()} non-positive values")

Open: 0 non-positive values
High: 0 non-positive values
Low: 0 non-positive values
Close: 0 non-positive values
Volume: 0 non-positive values


## 2. Save the Cleaned Dataset

The validation checks show that the raw SPY dataset does not require any
row removal or value correction.

We therefore preserve the observations unchanged and save the validated
dataset to `data/processed/spy_daily_clean.csv`.

This processed dataset will be used as the starting point for exploratory
analysis and feature engineering.

In [15]:
# Save the validated dataset

CLEAN_DATA_FILE = PROCESSED_DATA_DIR / "spy_daily_clean.csv"

spy_data.to_csv(CLEAN_DATA_FILE)

print(f"Cleaned data saved to: {CLEAN_DATA_FILE}")

Cleaned data saved to: /Users/amitkumar/Desktop/Computing/Projects/Finance/market-volatility/data/processed/spy_daily_clean.csv


verify it:

In [16]:
# Verify the saved cleaned dataset

print(f"File exists: {CLEAN_DATA_FILE.exists()}")
print(f"File size: {CLEAN_DATA_FILE.stat().st_size / 1024:.1f} KB")

File exists: True
File size: 382.8 KB


In [17]:
# Final dataset summary

print(f"Observations: {len(spy_data):,}")
print(f"Columns: {list(spy_data.columns)}")
print(f"Date range: {spy_data.index.min().date()} to {spy_data.index.max().date()}")

Observations: 4,191
Columns: ['Close', 'High', 'Low', 'Open', 'Volume']
Date range: 2010-01-04 to 2026-09-01


## 3. Notebook Summary
The raw SPY dataset was inspected and validated before being saved as a
processed dataset.

The following checks were performed:

- The date index is a `DatetimeIndex`.
- Observations are ordered chronologically.
- No duplicate dates are present.
- No missing values are present.
- No infinite numerical values are present.
- The OHLC price relationships are valid within a numerical tolerance.
- No non-positive prices or trading volumes are present.

No observations or values required correction or removal. The validated
dataset was saved to `data/processed/spy_daily_clean.csv`.

The cleaned dataset is now ready for exploratory data analysis and the
construction of volatility-related features.